In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session
import logging
logging.basicConfig(level=logging.INFO)


In [ ]:
!pip install transformers datasets peft accelerate bitsandbytes -q
!pip install git+https://github.com/huggingface/peft.git -q
!pip install -U bitsandbytes



In [ ]:
import pandas as pd
import json


df = pd.read_csv("/kaggle/input/business-strategy-dataset/business_strategy_dataset.csv")

formatted_data = []
for _, row in df.iterrows():
    instruction = f"""business_name: {row['business_name']}
industry: {row['industry']}
challenges: {row.get('challenges', '')}
goals: {row.get('goals', '')}
target_audience: {row.get('target_audience', '')}
timeframe: {row.get('timeframe', '')}
budget: {row.get('budget', '')}"""
    
    output = f"""response_title: {row['response_title']}
summary: {row['summary']}
strategies: {row['strategies']}
action_plan: {row['action_plan']}"""
    
    formatted_data.append({"instruction": instruction, "output": output})

with open("formatted_data.jsonl", "w") as f:
    for item in formatted_data:
        f.write(json.dumps(item) + "\n")
print(df.head())


In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0", 
    local_dir="tinyllama", 
    local_dir_use_symlinks=False
)


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

model = AutoModelForCausalLM.from_pretrained(
    "/kaggle/working/tinyllama",  
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained("/kaggle/working/tinyllama")
tokenizer.pad_token = tokenizer.eos_token


In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="formatted_data.jsonl", split="train")

def format(example):
    return {
        "input_ids": tokenizer(
            example["instruction"],
            truncation=True,
            padding="max_length",
            max_length=512
        )["input_ids"],
        "labels": tokenizer(
            example["output"],
            truncation=True,
            padding="max_length",
            max_length=512
        )["input_ids"]
    }

tokenized_dataset = dataset.map(format)
print(tokenized_dataset[0])



In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],  
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
if hasattr(model, "peft_config"):
    model.unload()

model = get_peft_model(model, lora_config)

In [ ]:
from transformers import (
    Trainer, TrainingArguments,
    AutoModelForSequenceClassification, AutoTokenizer,
    default_data_collator
)
from datasets import load_dataset

import os
os.environ["WANDB_MODE"] = "offline"  

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

raw_dataset = load_dataset("imdb")
train_dataset = raw_dataset["train"].select(range(20000))  
eval_dataset = raw_dataset["test"].select(range(5000))

def tokenize_fn(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=512
    )

train_dataset = train_dataset.map(tokenize_fn, batched=True)
eval_dataset = eval_dataset.map(tokenize_fn, batched=True)

train_dataset = train_dataset.rename_column("label", "labels")
eval_dataset = eval_dataset.rename_column("label", "labels")

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
eval_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    save_strategy="epoch",
    logging_steps=500,
    disable_tqdm=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    data_collator=default_data_collator
)

trainer.train()


In [ ]:
eval_results = trainer.evaluate()
print("Evaluation results:", eval_results)


In [ ]:
trainer.model.save_pretrained("./my_model")
tokenizer.save_pretrained("./my_model")


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("./my_model")
tokenizer = AutoTokenizer.from_pretrained("./my_model")

def generate_response(instruction):
    # Tokenize the instruction
    input_ids = tokenizer(instruction, return_tensors="pt").input_ids
    
    # Generate the output (response)
    output_ids = model.generate(input_ids, max_length=512, num_beams=5, early_stopping=True)
    
    # Decode the generated response
    response = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return response

instruction = """
business_name: XYZ Corp
industry: Technology
challenges: High competition in the market
goals: Increase market share by 15%
target_audience: Tech enthusiasts and businesses
timeframe: 12 months
budget: 1 million USD
"""


response = generate_response(instruction)
print("Generated Response:", response)
